# 07 · Visual-token compression (stage S4)

Every method keeps a fraction `k` of the visual tokens before the decoder, through one call, `api.compress`, chosen by
`cfg["compressor"]`. This notebook checks each method on one GQA and one TextVQA dev item. Nothing here is a result:
it shows that the kept count is exact, that `k = 100%` changes nothing, and what the model answers.

| Method | Chooses tokens by | Code |
|---|---|---|
| `random`, `uniform`, `text_mean` | chance / even spacing / similarity to the mean question embedding | ours (controls) |
| `lowres` | nothing: a smaller image with about `k` times the tokens | config only (control) |
| `crisp_lite` | similarity to the question's nouns, plus a uniform fill | reimplemented from the CRISP paper |
| `rediprune` | relevance to the question and diversity | official |
| `divprune` | diversity only | official |
| `visionzip` | attention in the vision tower; the rest is merged | official |

Details and deviations: `docs/notes/compression_baselines.md`.

**How to run:** Run All, no edits, then paste the Step-4 output. One model copy is used. Needs spaCy with `en_core_web_sm`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# GPU choice must precede CUDA init, which importing adaloop.api already does: the one import outside adaloop.api
from utils.gpus import select_for_notebook

CONFIG = "configs/mvp.yaml"
GPU_IDS, N_COPIES = select_for_notebook(CONFIG)

In [ ]:
import os
import socket

In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook("configs/mvp.yaml")
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
import torch

IS_INARA = "inara" in socket.gethostname()
print("host", socket.gethostname(), "| on Inara:", IS_INARA)
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
for i in range(torch.cuda.device_count()):
    total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i} | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")

In [ ]:
import pandas as pd
import spacy

In [ ]:
# Settings
METHODS = [m for m in api.METHODS if m != "none"]
KS = [0.25, 0.5]  # keep fractions checked here; 100% is checked separately
R = 3  # loop passes
NOUN_MODEL = cfg["compression"]["crisp_lite"]["spacy_model"]
METHODS, cfg["compression"]

### Step-1:- Load the model and the noun tagger

`crisp_lite` needs spaCy's small English model; loading it first fails fast if it is missing.

In [ ]:
print("spaCy", spacy.__version__, "|", [t.text for t in spacy.load(NOUN_MODEL)("What color is the car?") if t.pos_ == "NOUN"])

In [ ]:
print("[STAGE] load model")
vlm = api.load_models(cfg, N_COPIES)[0]
api.set_loops(vlm, R)

### Step-2:- Two items and the uncompressed reference

The reference is the model with every visual token and no compression step.

In [ ]:
items = [api.load_gqa(cfg, "dev", limit=1, root=ROOT)[0], api.load_textvqa(cfg, "dev", limit=1, root=ROOT)[0]]
[(it.benchmark, it.question, it.answers[:3]) for it in items]

In [ ]:
def run(it, method, k):
    """Compress one item with `method` at keep fraction `k`, then generate."""
    run_cfg = {**cfg, "compressor": method}
    inputs, info = api.compress(vlm, api.item_inputs(vlm, it, run_cfg), it, k, run_cfg)
    return api.generate_answers(vlm, [inputs], run_cfg)[0], info


print("[STAGE] reference")
reference = {}
for it in items:
    gen, info = run(it, "none", 1.0)
    reference[it.item_id] = gen
    print(f"{it.benchmark} | visual tokens {info['n_vis']} (grid {info['grid_hw']}) | raw {gen.raw_text!r} | "
          f"pred {api.read_answer(gen, cfg)!r} | gold {it.answers[:3]}")

### Step-3:- k = 100% changes nothing

At `k = 100%` a method keeps everything but still takes its packed path (vision tower run outside `generate`), so equal
token ids check the plumbing.

In [ ]:
print("[STAGE] k = 100%")
for method in METHODS:
    for it in items:
        gen, info = run(it, method, 1.0)
        same = torch.equal(gen.token_ids, reference[it.item_id].token_ids)
        assert same and info["n_vis"] == info["n_vis_full"], f"{method} {it.benchmark}: k=1 changed the output {gen.raw_text!r}"
    print(f"✓ {method}: same generated ids as no compression on both items")

### Step-4:- Each method at 25% and 50% (paste this output)

Per method, `k` and item: kept tokens of the full count, the raw output, the parsed answer and the references.
`lowres` cannot hit an exact count (the grid grows a row or column at a time).

In [ ]:
print("[STAGE] methods")
rows = []
for method in METHODS:
    for k in KS:
        for it in items:
            gen, info = run(it, method, k)
            pred = api.read_answer(gen, cfg)
            rows.append({"method": method, "k": k, "benchmark": it.benchmark, "n_vis": info["n_vis"],
                         "n_vis_full": info["n_vis_full"], "grid_hw": info["grid_hw"], "res_px": info["res_px"],
                         "pred": pred, "correct": api.score(it.benchmark, pred, it.answers),
                         "vit_s": info["vit_s"], "compression_s": info["compression_s"],
                         "compression_flops": info["compression_flops"], "fallback": info["fallback"]})
            note = f" | fallback {info['fallback']}" if info["fallback"] else ""
            print(f"{method:10s} k={k:.2f} {it.benchmark:7s} | kept {info['n_vis']}/{info['n_vis_full']}{note}")
            print(f"    raw {gen.raw_text!r} | pred {pred!r} | gold {it.answers[:3]}")
table = pd.DataFrame(rows)

In [ ]:
table[["method", "k", "benchmark", "n_vis", "n_vis_full", "res_px", "correct", "vit_s", "compression_s", "compression_flops"]]

In [ ]:
for r in table.itertuples():
    want = api.n_keep_for(r.n_vis_full, r.k)
    if r.method == "lowres":
        assert abs(r.n_vis - want) <= max(r.grid_hw), f"lowres {r.benchmark} k={r.k}: {r.n_vis} vs {want}, grid {r.grid_hw}"
    else:
        assert r.n_vis == want, f"{r.method} {r.benchmark} k={r.k}: kept {r.n_vis}, want {want}"
print(f"✓ kept counts: exact for every method at {KS} (lowres within one grid row)")